# Teste do ambiente

Este notebook só confere se o seu computador está pronto para o minicurso.

**Como rodar:** menu **Ambiente de execução → Executar tudo**. Leva cerca de 1 minuto.

No final, a última célula deve mostrar **✅ Tudo certo**. Você não precisa entender o código agora.

## 1. O Colab está ligado?

In [ ]:
import sys

print("Olá, Colab!")
print("Versão do Python:", sys.version.split()[0])

## 2. Instalar as ferramentas de otimização

- **Pyomo:** a linguagem em que escrevemos o modelo.
- **HiGHS:** o *solver*, o programa que encontra a melhor solução.

O Colab apaga tudo quando você fecha a aba, então esta instalação roda toda vez que o notebook é aberto.

In [ ]:
%pip install -q pyomo highspy

## 3. Um modelo bem pequeno

Duas variáveis, duas restrições e um objetivo:

$$
\begin{aligned}
\max \quad & 5x + 4y \\
\text{sujeito a} \quad & 6x + 4y \le 24 \\
& x + 2y \le 6 \\
& x,\ y \ge 0
\end{aligned}
$$

A resposta certa é $x = 3$, $y = 1{,}5$ e objetivo $= 21$.

In [ ]:
import pyomo.environ as pyo

modelo = pyo.ConcreteModel("teste")

# variáveis: não negativas
modelo.x = pyo.Var(domain=pyo.NonNegativeReals)
modelo.y = pyo.Var(domain=pyo.NonNegativeReals)

# objetivo: maximizar 5x + 4y
@modelo.Objective(sense=pyo.maximize)
def objetivo(m):
    return 5 * m.x + 4 * m.y

# restrição 1: 6x + 4y <= 24
@modelo.Constraint()
def restricao_1(m):
    return 6 * m.x + 4 * m.y <= 24

# restrição 2: x + 2y <= 6
@modelo.Constraint()
def restricao_2(m):
    return m.x + 2 * m.y <= 6

## 4. Resolver e conferir

In [ ]:
def resolve_e_confere(modelo):
    solver = pyo.SolverFactory("appsi_highs")
    resultado = solver.solve(modelo)
    status = resultado.solver.termination_condition

    x, y, fo = pyo.value(modelo.x), pyo.value(modelo.y), pyo.value(modelo.objetivo)
    print("Status:", status)
    print(f"x = {x:.1f} · y = {y:.1f} · objetivo = {fo:.1f}")

    if str(status) == "optimal" and abs(fo - 21) < 1e-6:
        print("✅ Tudo certo: o ambiente está pronto.")
    else:
        print("⚠️ Algo não saiu como esperado. Chame o professor.")

resolve_e_confere(modelo)